# 00 — Setup and First Capture

**Purpose:** Establish the baseline: import torchlens, run the first trace, and explore the
top-level ergonomics every user encounters in the first five minutes.

**Surfaces covered:**
- [ ] `import torchlens as tl` + `tl.__version__`
- [ ] `tl.trace(model, x)` — basic forward capture
- [ ] `Trace.__repr__` and `Trace.__str__`
- [ ] `Trace.summary()` — default and named levels
- [ ] `trace[label]` — first activation access via string indexing
- [ ] `Op.out` — tensor payload on a record
- [ ] `trace.layer_labels` and `trace.op_labels`
- [ ] `tl.peek(model, x, layer)` — one-layer extract convenience

## 1. Environment setup

The notebook imports the shared `_models` zoo from the same directory (notebooks/audit/).  
No network required; all models are tiny CPU tensors.

In [ ]:
import pathlib
import sys

# Ensure _models.py (in the same directory as this notebook) is importable
sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")
print(f"ZOO entries       : {list(ZOO.keys())}")

## 2. Import sanity — `tl.__all__`

Verify the expected public names are exported. Any missing name here is a GAP.

In [ ]:
expected_top_level = [
    "trace",
    "peek",
    "extract",
    "batched_extract",
    "Container",
    "Trace",
    "Layer",
    "Op",
    "Module",
    "Param",
    "Buffer",
    "GradFn",
    "record",
    "fastlog",
    "save",
    "load",
    "validate",
    "Bundle",
    "bundle",
    "do",
    "replay",
    "replay_from",
    "rerun",
    "sweep",
    "sites",
    "tap",
    "record_span",
    "facets",
    "facet",
    "head",
    "func",
    "module",
    "in_module",
    "output",
    "where",
    "followed_by",
    "preceded_by",
    "when",
    "add",
    "replace_with",
    "zero_ablate",
    "mean_ablate",
    "scale",
    "steer",
]

present = [n for n in expected_top_level if hasattr(tl, n)]
missing = [n for n in expected_top_level if not hasattr(tl, n)]

print(f"Present : {len(present)}/{len(expected_top_level)}")
if missing:
    print(f"MISSING : {missing}")
else:
    print("All expected names present.")

print(f"\ntl.__all__ has {len(tl.__all__)} entries total.")
print(sorted(tl.__all__))

## 3. `tl.trace` — basic forward capture

Run the simplest possible capture on `tiny_mlp` (8->relu->4 linear chain).  
This exercises the core entry point that every user calls first.

In [ ]:
model, x = ZOO["tiny_mlp"]()

trace = tl.trace(model, x)

print("type:", type(trace).__name__)
print("repr:", repr(trace))

## 4. `Trace.__str__` — the human-readable log string

`repr` is the short one-liner; `str` gives the full structured log text.

In [ ]:
print(str(trace))

## 5. `Trace.summary()` — overview and named levels

`summary()` returns a formatted string (not printed automatically — call `print`).  
Available levels: `'overview'` (default), `'graph'`, `'memory'`, `'control_flow'`,
`'compute'`, `'cost'`, `'waterfall'`, `'output'`.

In [ ]:
# Default level
print(trace.summary())

In [ ]:
# graph level — topology-oriented summary
print(trace.summary(level="graph"))

In [ ]:
# memory level — allocation/tensor-size oriented
print(trace.summary(level="memory"))

## 6. Label accessors — `layer_labels` and `op_labels`

`layer_labels` are the stable human-readable keys (used in most indexing).  
`op_labels` are pass-qualified (`label:pass`) and uniquely identify each op execution.

In [ ]:
print("layer_labels :", trace.layer_labels)
print("op_labels    :", trace.op_labels)

## 7. First activation access — `trace[label]` and `Op.out`

Indexing the trace by label returns the `Layer` (or `Op`) record for that operation.  
The `.out` attribute holds the captured output tensor.

In [ ]:
# Integer indexing
first = trace[0]
print("trace[0] type  :", type(first).__name__)
print("trace[0] repr  :")
print(repr(first))

In [ ]:
# String label indexing — use a real label from the trace
relu_label = trace.layer_labels[2]  # relu op
relu_op = trace[relu_label]
print(f"trace['{relu_label}'] repr:")
print(repr(relu_op))
print()
print("relu_op.out      :", relu_op.out)
print("relu_op.out.shape:", relu_op.out.shape)

In [ ]:
# Substring lookup — TorchLens resolves unambiguous substrings
try:
    found = trace["relu"]
    print("trace['relu'] label :", found.layer_label)
    print("trace['relu'] shape :", found.out.shape)
except Exception as exc:
    print(f"Substring lookup error: {exc}")

## 8. Top-level accessors — `trace.layers`, `trace.ops`, `trace.modules`, `trace.params`

Each accessor gives a structured view of a different entity type.

In [ ]:
print("--- trace.layers ---")
print(repr(trace.layers))
print()
print("--- trace.ops ---")
print(repr(trace.ops))
print()
print("--- trace.modules ---")
print(repr(trace.modules))
print()
print("--- trace.params ---")
print(repr(trace.params))

## 9. `tl.peek` — one-layer extract convenience

`tl.peek(model, x, layer)` runs a trace, extracts the tensor for one layer, and returns it.
Useful when the caller only wants a single activation without keeping the whole trace.

In [ ]:
model, x = ZOO["tiny_mlp"]()

# Use a known layer label
target_layer = "relu_1_2"

activation = tl.peek(model, x, target_layer)
print(f"tl.peek result for '{target_layer}':")
print("  type  :", type(activation).__name__)
print("  shape :", activation.shape)
print("  dtype :", activation.dtype)
print("  value :", activation)

## 10. A second model — `demo_model` (kitchen sink)

Show `trace.summary()` on a more complex model to illustrate how the summary
scales: conditional branch, loop submodule, buffer, cos op.

In [ ]:
model2, x2 = ZOO["demo_model"]()
trace2 = tl.trace(model2, x2)

print(repr(trace2))
print()
print(trace2.summary())

---

## ⚠️ GAPs / ergonomic smells

- **`Trace.summary()` levels appear identical in output** for `'overview'` and `'graph'` levels
  on a simple model — both return the same block text. It is unclear whether the level kwarg
  has per-level sections that only differ on more complex models, or whether the level
  parameter is partially implemented. Worth JMT checking that each named level is
  meaningfully distinct.
- **`trace[0]` returns a `Layer` not an `Op`** — the `repr` header says "Layer" with
  "operation N/M" sub-info, which may be confusing to users who expect an `Op` from integer
  indexing. Worth clarifying in docs.
- No issues with `tl.peek`, indexing, `layer_labels`/`op_labels`, or accessor reprs —
  all ergonomically clean.